# Exercices de la Rencontre 10 : Pandas (avancé) et Matplotlib (bases)

Ce notebook regroupe les **exercices de création** de la rencontre 10. Vous y pratiquerez :

- la sélection avec `loc` et `iloc`, le filtrage booléen et le tri (`sort_values`) ;
- le nettoyage (`dropna`, `fillna`, `drop_duplicates`) et la transformation (nouvelles colonnes, `apply`, `map`) ;
- les agrégations avec `groupby` (`mean`, `sum`, `count`) ;
- les graphiques de base avec Matplotlib **sans passer par Pandas** (`plot`, `scatter`, `hist`, `bar`).

Essayez d'abord chaque exercice par vous-même avant de consulter la solution proposée.

### Accès aux jeux de données

Exécutez la cellule suivante en premier. La fonction `chemin_donnees()` cherche d'abord le fichier CSV dans le même dossier que le notebook (comme demandé dans les exercices du cours). Si le fichier n'y est pas (par exemple dans Google Colab), elle utilise la copie en ligne du dépôt du cours.

In [ ]:
import os

URL_DONNEES = "https://raw.githubusercontent.com/Simonll/420-SN1/develop/web/static/ressources/csv/"


def chemin_donnees(nom_fichier):
    """Retourne le chemin local du fichier s'il existe, sinon son adresse en ligne."""
    for dossier in [".", "../../web/static/ressources/csv"]:
        chemin = os.path.join(dossier, nom_fichier)
        if os.path.exists(chemin):
            return chemin
    return URL_DONNEES + nom_fichier


print(chemin_donnees("tips_dataset.csv"))

## Exercices de création

### Exercice 1 - Analyse complète d'un jeu de données avec Pandas

Utilisez le fichier `Turing.csv`, qui contient les lauréats du prix Turing (colonnes `year`, `laureate`, `contribution`, `affiliation`).

**Exploration**

1. Chargez les données dans un DataFrame ;
2. Affichez le nombre de lignes et de colonnes (`shape`) ;
3. Affichez la liste des colonnes ;
4. Affichez les types de données (`dtypes`) ;
5. Affichez le résumé technique (`info()`) ;
6. Affichez les statistiques descriptives (`describe()`).

In [ ]:
import pandas as pd

df_turing = pd.read_csv(chemin_donnees("Turing.csv"))

print("Forme :", df_turing.shape)
print("Colonnes :", list(df_turing.columns))
print("\nTypes :")
print(df_turing.dtypes)
print()
df_turing.info()
print()
print(df_turing.describe())

**Filtrage et tri**

7. Affichez les lauréats dont la contribution contient le mot « intelligence » ;
8. Affichez les lauréats affiliés à MIT ou à Stanford ;
9. Affichez les lauréats ayant reçu le prix entre 1980 et 1990 inclusivement ;
10. Affichez les 5 lauréats les plus récents (tri par année décroissante).

Résultat attendu pour les lauréats avec « intelligence » :

```
Lauréats avec 'intelligence' dans la contribution :
    - Marvin Minsky
    - Allen Newell
    - Herbert A. Simon
    - Raj Reddy
```

In [ ]:
masque_ia = df_turing["contribution"].str.contains("intelligence")
print("Lauréats avec 'intelligence' dans la contribution :")
for nom in df_turing.loc[masque_ia, "laureate"]:
    print(f"    - {nom}")

masque_mit_stanford = (df_turing["affiliation"] == "MIT") | (df_turing["affiliation"] == "Stanford University")
print("\nLauréats affiliés à MIT ou Stanford :")
print(df_turing.loc[masque_mit_stanford, ["year", "laureate", "affiliation"]])

masque_80s = (df_turing["year"] >= 1980) & (df_turing["year"] <= 1990)
print("\nLauréats de 1980 à 1990 :")
print(df_turing.loc[masque_80s, ["year", "laureate"]])

print("\nLes 5 lauréats les plus récents :")
print(df_turing.sort_values("year", ascending=False).head(5)[["year", "laureate"]])

**Agrégation et statistiques**

11. Créez une nouvelle colonne `decennie` = `(year // 10) * 10` ;
12. Affichez le nombre de lauréats par décennie (`groupby` + `count`) ;
13. Affichez le nombre de lauréats par affiliation (`groupby` + `count`).

Résultat attendu pour les lauréats par décennie :

```
Lauréats par décennie :
    - 1960s : 4 lauréats
    - 1970s : 12 lauréats
    - 1980s : 12 lauréats
    - 1990s : 12 lauréats
    - 2000s : 16 lauréats
    - 2010s : 16 lauréats
    - 2020s : 7 lauréats
```

In [ ]:
df_turing["decennie"] = (df_turing["year"] // 10) * 10

par_decennie = df_turing.groupby("decennie")["laureate"].count()
print("Lauréats par décennie :")
for decennie, nombre in par_decennie.items():
    print(f"    - {decennie}s : {nombre} lauréats")

par_affiliation = df_turing.groupby("affiliation")["laureate"].count()
print("\nLauréats par affiliation (les plus fréquentes) :")
print(par_affiliation.sort_values(ascending=False).head(10))

### Exercice 2 - Nettoyage et sélection

Un registraire de cégep dispose d'un tableau d'étudiants en Sciences de la nature contenant des données manquantes et des doublons. Utilisez le dictionnaire fourni pour créer votre DataFrame.

**Nettoyage**

1. Affichez les lignes ne contenant aucune valeur manquante ;
2. Remplacez les valeurs manquantes : `session` par la moyenne, `note_finale` par la médiane, `nb_absences` par 0 ;
3. Supprimez les lignes dupliquées.

**Sélection**

4. À partir du DataFrame corrigé (avant suppression des doublons), affichez le prénom et la note finale des étudiants en 200.BA ;
5. Affichez les 3 premières lignes et les colonnes `prenom`, `programme`, `session`.

Résultat attendu après suppression des doublons :

```
    prenom programme  session  note_finale  nb_absences
0      Léa    200.BA      3.0         78.0          2.0
1   Mathis    200.BB      2.6         85.0          0.0
3  Yasmine    200.BC      2.0         78.0          5.0
4  Tristan    200.BA      2.6         91.0          0.0
5      NaN    200.BB      4.0         67.0          1.0
6    Émile    200.BC      1.0         78.0          3.0
```

In [ ]:
data = {
    'prenom':      ['Léa', 'Mathis', 'Léa', 'Yasmine', 'Tristan', None, 'Émile'],
    'programme':   ['200.BA', '200.BB', '200.BA', '200.BC', '200.BA', '200.BB', '200.BC'],
    'session':     [3, None, 3, 2, None, 4, 1],
    'note_finale': [78, 85, 78, None, 91, 67, None],
    'nb_absences': [2, 0, 2, 5, None, 1, 3]
}
df_etudiants = pd.DataFrame(data)

print("1. Lignes complètes :")
print(df_etudiants.dropna())

df_corrige = df_etudiants.fillna({
    "session": df_etudiants["session"].mean(),
    "note_finale": df_etudiants["note_finale"].median(),
    "nb_absences": 0,
})

df_sans_doublons = df_corrige.drop_duplicates()
print("\n3. Après suppression des doublons :")
print(df_sans_doublons)

print("\n4. Étudiants en 200.BA :")
print(df_corrige.loc[df_corrige["programme"] == "200.BA", ["prenom", "note_finale"]])

print("\n5. Trois premières lignes, trois colonnes :")
print(df_corrige.iloc[:3].loc[:, ["prenom", "programme", "session"]])

### Exercice 3 - Visualisation Matplotlib : pourboire par personne

Utilisez le fichier `tips_dataset.csv`. Les graphiques doivent être faits **avec Matplotlib, sans les fonctions de graphique de Pandas**.

1. Chargez les données avec Pandas ;
2. Extrayez les colonnes `total_bill`, `tip` et `size` avec `.to_numpy()` (ou `.values`) ;
3. Calculez `tip_par_person` = `tip / size` à partir de ces tableaux ;
4. Tracez un histogramme des valeurs de `tip_par_person` avec 15 bins ;
5. Ajoutez le titre « Distribution du pourboire par personne », le label x « Pourboire par personne ($) » et le label y « Nombre de repas » ;
6. Ajoutez une grille ;
7. Sauvegardez dans `tip_par_person_histogram.png`.

In [ ]:
import matplotlib.pyplot as plt

df_tips = pd.read_csv(chemin_donnees("tips_dataset.csv"))

total_bill = df_tips["total_bill"].to_numpy()
tip = df_tips["tip"].to_numpy()
size = df_tips["size"].to_numpy()

tip_par_person = tip / size

plt.hist(tip_par_person, bins=15, color="steelblue", edgecolor="white")
plt.title("Distribution du pourboire par personne")
plt.xlabel("Pourboire par personne ($)")
plt.ylabel("Nombre de repas")
plt.grid(True, alpha=0.3)
plt.savefig("tip_par_person_histogram.png")
plt.show()

8. Tracez un nuage de points (*scatter plot*) de `tip` en fonction de `total_bill` avec les tableaux extraits ;
9. Ajoutez le titre « Pourboire vs Facture », le label x « Facture ($) » et le label y « Pourboire ($) » ;
10. Ajoutez une grille ;
11. Sauvegardez dans `tips_scatter.png`.

In [ ]:
plt.scatter(total_bill, tip, color="darkorange", alpha=0.7)
plt.title("Pourboire vs Facture")
plt.xlabel("Facture ($)")
plt.ylabel("Pourboire ($)")
plt.grid(True, alpha=0.3)
plt.savefig("tips_scatter.png")
plt.show()

### Exercice 4 - Synthèse : moyennes et interprétation

Toujours avec `tips_dataset.csv` :

1. Calculez la facture moyenne par jour de la semaine (`groupby()` et `mean()`) ;
2. Calculez le total des pourboires par moment de la journée (`time`) avec `groupby()` et `sum()` ;
3. Calculez la taille moyenne du groupe par jour avec `groupby()` et `mean()` ;
4. Comptez le nombre de repas par sexe avec `groupby()` et `count()` ;
5. Calculez la facture moyenne par statut fumeur (`smoker`) avec `groupby()` et `mean()`.

Résultat attendu pour la facture moyenne par jour :

```
day
Fri     17.151579
Sat     20.441379
Sun     21.410000
Thur    17.682742
Name: total_bill, dtype: float64
```

In [ ]:
print("1. Facture moyenne par jour :")
print(df_tips.groupby("day")["total_bill"].mean())

print("\n2. Total des pourboires par moment de la journée :")
print(df_tips.groupby("time")["tip"].sum())

print("\n3. Taille moyenne du groupe par jour :")
print(df_tips.groupby("day")["size"].mean())

print("\n4. Nombre de repas par sexe :")
print(df_tips.groupby("sex")["total_bill"].count())

print("\n5. Facture moyenne par statut fumeur :")
print(df_tips.groupby("smoker")["total_bill"].mean())

En observant ces résultats, répondez aux questions suivantes dans la cellule ci-dessous (en commentaires) :

- Quel jour de la semaine génère les factures les plus élevées ? Et les plus faibles ?
- Le souper (*Dinner*) ou le dîner (*Lunch*) génère-t-il plus de pourboires au total ? Pourquoi, selon vous ?
- Les fumeurs dépensent-ils plus en moyenne que les non-fumeurs ?
- Quel jour vient-on en plus grand groupe ? Est-ce cohérent avec la facture moyenne de ce jour ?
- Les hommes ou les femmes viennent-ils manger plus souvent au restaurant ?

In [ ]:
# Exemple de réponses (à comparer avec les vôtres) :
# - Le dimanche (Sun) a la facture moyenne la plus élevée (environ 21,41 $),
#   le vendredi (Fri) la plus faible (environ 17,15 $).
# - Le souper (Dinner) génère beaucoup plus de pourboires au total : il y a plus
#   de repas le soir dans ce jeu de données et les factures y sont plus élevées.
# - Oui, légèrement : environ 20,76 $ pour les fumeurs contre 19,19 $ pour les non-fumeurs.
# - Le dimanche, avec des groupes d'environ 2,84 personnes en moyenne. C'est cohérent :
#   c'est aussi le jour où la facture moyenne est la plus élevée.
# - Les hommes : 157 repas contre 87 pour les femmes dans ce jeu de données.

### Exercice 5 - Diagramme à barres et conversion de dates

**Partie 1 - Diagramme à barres**

À partir de petites listes de ventes par mois (par exemple `mois = ['Jan', 'Fév', 'Mar']` et `ventes = [120, 150, 90]`), utilisez `plt.bar(mois, ventes)` pour produire un **diagramme à barres** (et non une courbe). Ajoutez un titre, les labels d'axes et une légende si c'est pertinent.

In [ ]:
mois = ["Jan", "Fév", "Mar", "Avr"]
ventes_2024 = [120, 150, 90, 130]

plt.bar(mois, ventes_2024, color="seagreen", label="Ventes 2024")
plt.title("Ventes mensuelles")
plt.xlabel("Mois")
plt.ylabel("Nombre de ventes")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.show()

**Partie 2 - Conversion de dates avec `to_datetime`**

Créez un DataFrame avec une colonne `date` contenant des chaînes comme `'2024-01-15'`. Convertissez-la en véritable date avec `pd.to_datetime(df['date'])` et stockez le résultat dans une nouvelle colonne `date_obj`. Affichez le type (`dtype`) de la nouvelle colonne pour confirmer qu'il s'agit bien de dates (et non de texte).

Résultat attendu :

```text
dtype de date_obj : datetime64[ns]
```

Selon la version de Pandas installée, vous pourriez voir `datetime64[us]` plutôt que `datetime64[ns]` : seule la précision change, l'important est que le type soit `datetime64`.

In [ ]:
df_ventes = pd.DataFrame({
    "date": ["2024-01-15", "2024-02-15", "2024-03-15", "2024-04-15"],
    "ventes": ventes_2024,
})

df_ventes["date_obj"] = pd.to_datetime(df_ventes["date"])

print("dtype de date :", df_ventes["date"].dtype)
print("dtype de date_obj :", df_ventes["date_obj"].dtype)

# Une fois convertie, la date permet d'extraire facilement le mois
print(df_ventes["date_obj"].dt.month.tolist())

## Poursuite du TP2

Poursuivez votre TP2 avec les notions vues aujourd'hui :

- utilisez au moins 2 techniques Pandas avancées (filtres complexes, transformations, `groupby`) ;
- créez au moins 1 visualisation avec Matplotlib, sans les graphiques de Pandas (extrayez les données avec `.to_numpy()` ou `.values`) ;
- documentez vos choix techniques dans votre rapport.

Le TP2 n'est pas à remettre aujourd'hui.